# 03 — Training runner, pre-configured: `c1_ce_s43` (E1 seed replicate)

A copy of the `03_train` template with `RUN` pre-set — open, run all,
no edits needed. One Colab session per notebook; the sibling notebook
runs the other replicate in parallel.

**Why this run exists (E1, amendment to §0 rule 5 — see
`splits/CHANGELOG.md`):** identical config to the seed-42 run except
the seed, so the difference between the two runs measures the
pipeline's seed noise floor — the empirical calibration of the §0.5
"~2 points = comparable" band.

**C1_s43 extra duty:** its `best.ckpt` features also serve as the E2
concat control (C1 (+) C1'), cached in the diagnostics session -
training-side nothing changes.

Archive the executed copy under `notebooks/runs/` as
`YYYY-MM-DD_c1_ce_s43.ipynb`, as for every run (cell at the bottom).


## Environment setup

This notebook is not one run per file. Instead:

- one notebook = one runner
- one config = one experiment
- different people can use the same notebook with different config values

In Colab, the notebook runs on a remote VM. If the repo is not already
available there, this notebook will clone it from GitHub. It also stages
data from the shared Drive archive into `/content/data` using the
paths defined in `configs/paths.yaml`.


In [1]:
from pathlib import Path
import subprocess
import sys
import torch
import zipfile

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

# Locate or clone the repository root containing the sharp_har package.
# In Colab, the notebook may run from a temporary working directory.
cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.train import train_run
from sharp_har.utils import read_yaml

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
CKPT_ROOT = Path(paths_cfg["ckpt_root"])

# Mount Drive unconditionally (idempotent): checkpoints are written to
# CKPT_ROOT on Drive even when the data is already staged on the VM.
drive.mount("/content/drive")

# Stage the zip archives if needed (same convention as 00_setup_smoke).
if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)

print("Repo dir:", REPO_DIR)
print("sharp_har exists:", (REPO_DIR / "sharp_har").exists())
print("GPU available:", torch.cuda.is_available())
print("Stage dir:", stage_dir)
print("Checkpoint root:", CKPT_ROOT)


Mounted at /content/drive
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip -> /content/doppler_traces.zip
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces_S4_S5.zip -> /content/doppler_traces_S4_S5.zip
Repo dir: /content/sharp-har
sharp_har exists: True
GPU available: True
Stage dir: /content/data
Checkpoint root: /content/drive/MyDrive/sharp_har_runs


In [2]:
# GPU sanity check - run BEFORE launching training (see the C3 OOM incident:
# a stale process on the VM held 11.5 GiB). Expected on a clean runtime:
# no processes listed, ~0 MiB used. If not: Runtime -> Disconnect and
# delete runtime (a plain restart is NOT enough), then rerun from the top.
!nvidia-smi

import torch
free, total = torch.cuda.mem_get_info()
print(f"free: {free/2**30:.2f} GiB / total: {total/2**30:.2f} GiB")
assert free / 2**30 > 10, (
    "GPU is not clean: less than 10 GiB free before training. "
    "Something else is holding memory - delete the runtime and start over."
)
print("GPU clean - OK to launch.")

Sat Jul 18 07:46:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Load config and launch the run

Choose one config stem below and run the cell. The config name
determines which experiment is executed.


In [ ]:
RUN = "c1_ce_s43"  # E1 seed replicate - do not change in-session
cfg = read_yaml(REPO_DIR / "configs" / f"{RUN}.yaml")

print("Selected run:", RUN)
print("Config summary:")
print(cfg)

out = train_run(cfg, stage_dir=stage_dir, ckpt_dir=CKPT_ROOT, repo_dir=REPO_DIR)
print("Train run finished:", out)


2026-07-18 07:46:26,476 [INFO] sharp_har.data: train: 81 traces, 53400 (window, antenna) samples (win=340, stride=100)
2026-07-18 07:46:26,488 [INFO] sharp_har.data: val: 9 traces, 1396 (window, antenna) samples (win=340, stride=340)


Selected run: c1_ce_s43
Config summary:
{'name': 'C1_s43', 'seed': 43, 'protocol': 'P2-lab', 'split_file': 'splits/p2_lab.json', 'backbone': 'resnet_vb', 'd_enc': 256, 'n_att': 8, 'loss': {'type': 'ce', 'label_smoothing': 0.1}, 'adversary': {'type': None, 'target': 'ar_set', 'lambda_max': None, 'beta': None}, 'optim': {'name': 'adamw', 'lr': 0.001, 'wd': 0.0001, 'warmup_epochs': 5, 'scheduler': 'cosine'}, 'train': {'epoch_steps': 400, 'max_epochs': 40, 'batch_size': 256, 'sampler': 'uniform', 'amp': True, 'grad_clip': 1.0}, 'eval': {'select_metric': 'val_macro_f1', 'patience': 10}}


2026-07-18 07:46:34,748 [INFO] sharp_har.train: resumed C1_s43 from /content/drive/MyDrive/sharp_har_runs/C1_s43/last.ckpt at epoch 9
2026-07-18 07:50:30,509 [INFO] sharp_har.train: C1_s43 epoch 9/40: loss 0.9731, val macro-F1 0.4549, 0.583 s/step
2026-07-18 07:54:08,082 [INFO] sharp_har.train: C1_s43 epoch 10/40: loss 0.9482, val macro-F1 0.3545, 0.537 s/step
2026-07-18 07:57:40,793 [INFO] sharp_har.train: C1_s43 epoch 11/40: loss 0.9291, val macro-F1 0.6053, 0.525 s/step
2026-07-18 08:01:10,332 [INFO] sharp_har.train: C1_s43 epoch 12/40: loss 0.9065, val macro-F1 0.3389, 0.518 s/step
2026-07-18 08:04:39,097 [INFO] sharp_har.train: C1_s43 epoch 13/40: loss 0.8795, val macro-F1 0.5128, 0.516 s/step
2026-07-18 08:08:10,220 [INFO] sharp_har.train: C1_s43 epoch 14/40: loss 0.8557, val macro-F1 0.6942, 0.522 s/step
2026-07-18 08:11:38,780 [INFO] sharp_har.train: C1_s43 epoch 15/40: loss 0.8275, val macro-F1 0.6501, 0.514 s/step
2026-07-18 08:15:08,042 [INFO] sharp_har.train: C1_s43 epoch 1

## Training curves

Per-epoch diagnostics read from the run's `history.csv` (§0.4). The
panels adapt to the run: train loss and throughput always; fused val
macro-F1 with the val-selected best epoch on CE runs (C0/C1/C2); the
mandatory §6-C2 GRL monitoring pair (AR-set train accuracy vs λ ramp)
on C2/C4; learning rate. Re-run this cell any time — it re-reads the
CSV, so it also works on a resumed or still-running run.

Related helpers in `sharp_har.viz`: `plot_confusion(csv)` for the
harness `*_confusion.csv` files, `compare_runs({name: run_dir, ...})`
to overlay a metric across configs (remember §0.5: differences under
~2 points are "comparable", not improvements).


In [ ]:
# Thin runner: all plotting logic lives in sharp_har.viz.
from sharp_har.viz import plot_history

run_dir = CKPT_ROOT / cfg["name"]
plot_history(run_dir);


## Archiving the definitive run notebook

This notebook stays a **clean template** (outputs cleared on Git). The
executed copy of a real run is a measured artifact and is committed
verbatim, like the gate reports:

1. When the run (or a resumed segment) ends, download the executed
   notebook (`File → Download → .ipynb`) **with its outputs**.
2. Commit it under `notebooks/runs/` as `YYYY-MM-DD_<config>.ipynb`
   (e.g. `2026-07-16_c0_sharp.ipynb`); add a `_partN` suffix if one run
   spans several resumed sessions. Never edit archived outputs.
3. The run's numeric artifacts still live in `ckpt_root/<name>/`
   (`history.csv`, `run_meta.json`, checkpoints) — the archived
   notebook is the human-readable record, not a data source.

See `notebooks/runs/README.md` for the full convention.
